 # Unit Test Generator - Fine-tuning with Unsloth (ChatML Format)

 This script fine-tunes a Qwen2.5-Coder model for generating Python unit tests,

 specifically using the native ChatML template suitable for conversational models.

 ## 1. Load Base Model

In [1]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # Auto detection
load_in_4bit = True  # Use 4bit quantization to reduce memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Explicitly ensure we are using the model's native chat template
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5", # Standard chat template used by Qwen2.5 Instruct
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/sachithra/miniconda3/envs/unsloth_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.5: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.628 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


 ## 2. Add LoRA Adapters

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,  
    loftq_config = None, 
)


Unsloth 2025.12.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


 ## 3. Prepare Training Data

In [3]:
def formatting_prompts_func(examples):
    questions = examples["question"]
    codes     = examples["code_ground_truth"]
    unit_tests_lists = examples["unit_tests"]
    
    texts = []
    # Zip through the batch
    for question, code, tests_list in zip(questions, codes, unit_tests_lists):
        
        # safely extract the test suite
        if tests_list:
            if isinstance(tests_list, dict) and 'code' in tests_list:
                full_test_suite = "\n\n".join([str(c) for c in tests_list['code'] if c])
            elif isinstance(tests_list, list):
                full_test_suite = "\n\n".join([str(t['code']) for t in tests_list if isinstance(t, dict) and t.get('code')])
            elif isinstance(tests_list, str):
                full_test_suite = tests_list
            else:
                full_test_suite = ""
        else:
            full_test_suite = ""
            
        # Format the user input
        user_content = f"Write a comprehensive Python unit test suite for this code.\n\nProblem Description:\n{question}\n\nCode to Test:\n{code}"
        
        # Build the ChatML message structure
        messages = [
            {"role": "system", "content": "You are a helpful coding assistant that writes Python unit tests."},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": full_test_suite.strip()}
        ]
        
        # Apply the chat template (this also inherently adds EOS tokens where needed)
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
        
    return { "text" : texts }

from datasets import load_dataset

# Load both the train and test splits of the dataset
train_dataset = load_dataset("KAKA22/CodeRM-UnitTest", split="train")
test_dataset = load_dataset("KAKA22/CodeRM-UnitTest", split="test")

#Shuffle and take a smaller subset of the test dataset for faster evaluation
test_dataset = test_dataset.shuffle(seed=42).select(range(1000))

# Apply the formatting function to both datasets
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
test_dataset = test_dataset.map(formatting_prompts_func, batched=True)


Map: 100%|██████████| 1000/1000 [00:01<00:00, 672.57 examples/s]


 ## 4. Train the Model

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = test_dataset, 
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1, 
    args = SFTConfig(
        per_device_train_batch_size = 1, 
        per_device_eval_batch_size = 1,  
        dataset_num_proc = 1, 
        gradient_accumulation_steps = 8, 
        warmup_steps = 5,
        max_steps = 100,          
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "steps",  
        eval_steps = 20,          
        save_strategy = "steps",  
        save_steps = 20,          
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss", 
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2) 
    ]
)




[trl.trainer.sft_trainer|WARNING]You are using a per_device_train_batch_size of 1 with padding-free training. Using a batch size of 1 anihilate the benefits of padding-free training. Please consider increasing the batch size to at least 2.
Unsloth: Tokenizing ["text"] (num_proc=1):   6%|▌         | 1000/17562 [00:47<12:59, 21.24 examples/s]


TimeoutError: 

In [ ]:
trainer_stats = trainer.train()

 ## 5. Model Evaluation (Generative Metrics)

In [ ]:
import re 
import evaluate
from tqdm import tqdm

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

FastLanguageModel.for_inference(model)

predictions = []
references = []

eval_sample_size = 100
eval_subset = test_dataset.select(range(eval_sample_size))

print(f"Generating test cases for {eval_sample_size} samples...")
for i in tqdm(range(len(eval_subset))):
    sample = eval_subset[i]
    
    user_content = f"Write a comprehensive Python unit test suite for this code.\n\nProblem Description:\n{sample['question']}\n\nCode to Test:\n{sample['code_ground_truth']}"
    
    messages = [
        {"role": "system", "content": "You are a helpful coding assistant that writes Python unit tests."},
        {"role": "user", "content": user_content}
    ]
    
    # We use add_generation_prompt=True to append <|im_start|>assistant so the model knows it is its turn to write
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    
    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
    # Extract the assistant's response. In ChatML, skipping special tokens means we just need the newly generated text.
    # Since skip_special_tokens=True strips tags like <|im_start|>, we just use naive find/split. 
    # Or safely decode only new tokens:
    input_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_len:]
    response_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    # Try to extract just the python code between markdown ticks
    code_blocks = re.findall(r'```(?:python)?\n(.*?)\n```', response_text, re.DOTALL)
    if code_blocks:
        response_text = "\n\n".join(code_blocks).strip()
            
    # Extract ground truth safely
    try:
        ut = sample['unit_tests']
        if isinstance(ut, dict) and 'code' in ut:
            ground_truth = "\n\n".join([str(c) for c in ut['code'] if c]).strip()
        elif isinstance(ut, list):
            ground_truth = "\n\n".join([str(t['code']) for t in ut if isinstance(t, dict) and t.get('code')]).strip()
        elif isinstance(ut, str):
            ground_truth = ut.strip()
        else:
            ground_truth = ""
    except Exception as e:
        print(f"Failed to parse ground truth for a sample: {e}")
        ground_truth = ""
        
    predictions.append(response_text)
    references.append([ground_truth]) 

print("\n--- Evaluation Results ---")

filtered_predictions = []
filtered_references = []
empty_preds_count = 0

for pred, ref in zip(predictions, references):
    if not ref[0].strip():
        continue
        
    if not pred.strip():
        empty_preds_count += 1
        pred = "# empty" 
        
    filtered_predictions.append(pred)
    filtered_references.append(ref)

if empty_preds_count > 0:
    print(f"⚠️ Warning: The model generated empty responses for {empty_preds_count} out of {len(filtered_references)} valid samples.")

if len(filtered_references) == 0:
    print("Warning: All ground truth references were empty! Cannot compute BLEU/ROUGE.")
else:
    rouge_results = rouge_metric.compute(
        predictions=filtered_predictions, 
        references=[ref[0] for ref in filtered_references]
    )
    bleu_results = bleu_metric.compute(
        predictions=filtered_predictions, 
        references=filtered_references
    )

    print(f"BLEU Score:   {bleu_results['bleu'] * 100:.2f}%")
    print(f"ROUGE-1:      {rouge_results['rouge1'] * 100:.2f}%")
    print(f"ROUGE-2:      {rouge_results['rouge2'] * 100:.2f}%")
    print(f"ROUGE-L:      {rouge_results['rougeL'] * 100:.2f}%")



 ## 6. Save Merged Model

In [ ]:
output_dir = "merged_model_16bit"

model.save_pretrained_merged(output_dir, tokenizer, save_method="merged_16bit")
print(f"✅ Saved merged 16-bit Hugging Face model to '{output_dir}/'")
